# Neural Machine Translation Demo with Marian
## Sequence to Sequence Architecture

**By Ye Kyaw Thu, Lab Leader, Language Understanding Lab., Myanmar**  
**Date:** 24 May 2026  
*For AI (Fundamental) Class students*  

## Data Preparation

SMT assignment အတွက် သုံးထားတဲ့ grapheme to phoneme (g2p) ဒေတာကိုပဲ သုံးပြီး NMT architecture နှစ်မျိုးကိုသုံးပြီး machine translation လုပ်ပြပါမယ်။ စာကြောင်း အနေနဲ့က syllable ဖြတ်ထားတဲ့ word တွေမို့လို့ length သိပ်မရှည်ပါဘူး။ အဲဒါကြောင့် ဆရာ အခုသုံးပြမယ့် GPU machine မှာတော့ training time က မြန်ပါလိမ်မယ်။  



In [ ]:
%pwd

'/home/phantom/Documents/assignment_7_testing/kaung-htet-htun_assignment-7/notebooks'

In [2]:
!ls --color=auto

pdf  Seq2Seq-NMT-marian-ph2gp.ipynb  Transformer-NMT-marian-ph2gp.ipynb


In [4]:
%cd /home/phantom/Documents/assignment_7_testing/kaung-htet-htun_assignment-7/g2p-par

/home/phantom/Documents/assignment_7_testing/kaung-htet-htun_assignment-7/g2p-par


/home/phantom/Documents/AI_env/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [5]:
!ls *

dev.my	dev.ph	test.my  test.ph  train.my  train.ph


### Vocab Building  

Vocab ကို ဆောက်ဖို့အတွက် training ဖိုင်နဲ့ validation or development ဖိုင်ကို ပေါင်းပါမယ်။  
အဲဒီ အလုပ်ကို source language အတွက်ကော၊ target language အတွက်ကော လုပ်ရမယ်။  
အရင်ဆုံး preprocessing ဆိုတာဆောက်လိုက်ပြီး အဲဒီအထဲမှာ ပေါင်းထားတဲ့ ဖိုင်နှစ်ဖိုင်ကို သိမ်းကြရအောင်။  

In [6]:
!mkdir preprocessing

In [7]:
!cat train.my dev.my > ./preprocessing/train-dev.my

In [8]:
!cat train.ph dev.ph > ./preprocessing/train-dev.ph

In [9]:
!wc ./preprocessing/*.{my,ph}

 22000  63023 653405 ./preprocessing/train-dev.my
 22000  63034 286205 ./preprocessing/train-dev.ph
 44000 126057 939610 total


မြန်မာစာ အတွက် vocab ဖိုင်ကို ဆောက်မယ်။  

In [10]:
!mkdir vocab

In [15]:
!~/marian/build/marian-vocab < ./preprocessing/train-dev.my > ./vocab/vocab.my.yml

[2026-07-02 21:04:41] Creating vocabulary...
[2026-07-02 21:04:41] [data] Creating vocabulary stdout from stdin
[2026-07-02 21:04:41] Finished


Phoneme ပိုင်းအတွက်လည်း vocab ဖိုင်ကို ဆောက်ပါမယ်။  

In [16]:
!~/marian/build/marian-vocab < ./preprocessing/train-dev.ph > ./vocab/vocab.ph.yml

[2026-07-02 21:05:15] Creating vocabulary...
[2026-07-02 21:05:15] [data] Creating vocabulary stdout from stdin
[2026-07-02 21:05:15] Finished


Vocab ဖိုင်ရဲ့ format ကိုလေ့လာကြည့်ကြရအောင်။ တကယ်ကတော့ ကိုယ့် training/validation ဖိုင်ထဲမှာ ပါဝင်တဲ့ စာလုံးတစ်လုံးချင်းစီကို (i.e. unit words) နံပါတ်ထိုးပြီး သိမ်းထားတဲ့ ပုံစံပါပဲ။  

In [17]:
!head -n 30 ./vocab/vocab.my.yml 

</s>: 0
<unk>: 1
အ: 2
မ: 3
သ: 4
က: 5
တ: 6
လက်: 7
ပ: 8
စ: 9
စာ: 10
ရေ: 11
ရ: 12
ရာ: 13
ကြီး: 14
တစ်: 15
စား: 16
သား: 17
နာ: 18
ခ: 19
ခံ: 20
မီး: 21
လူ: 22
မျက်: 23
ကျ: 24
သာ: 25
သံ: 26
ပေါက်: 27
လုံး: 28
ကာ: 29


In [18]:
!head -n 30 ./vocab/vocab.ph.yml 

</s>: 0
<unk>: 1
a-: 2
ma-: 3
da-: 4
le': 5
jei: 6
ga-: 7
ta-: 8
tha-: 9
ja: 10
ba-: 11
dha-: 12
za-: 13
pa-: 14
na: 15
"mi:": 16
mje': 17
ka-: 18
than: 19
"gyi:": 20
za: 21
lu: 22
ja.: 23
ma.: 24
le: 25
sa: 26
na-: 27
kha-: 28
"loun:": 29


## Preparing a Shell Script for Phoneme to Myanmar Translation

Phoneme to Myanmar conversion ကို neural machine translation နဲ့ လုပ်ကြည့်ပြီး performance ကို တိုင်းတာကြည့်ကြရအောင်။  

အဲဒီလို လုပ်ဖို့အတွက် ဆရာကတော့ shell script ကို ပြင်ပြီး run တဲ့ ပုံစံနဲ့ပဲ သွားပါတယ်။  
Source language ကို phoneme ထားပြီး၊ target language ကို grapheme ထားမယ်။  

အရင်ဆုံး sequence to sequence အာခီတက်ချာနဲ့ပဲ သွားမယ်။  

In [21]:
%cd /home/phantom/Documents/assignment_7_testing/kaung-htet-htun_assignment-7

/home/phantom/Documents/assignment_7_testing/kaung-htet-htun_assignment-7


In [22]:
from IPython.display import Markdown

# Read the file content
with open('./seq2seq.phmy.sh', 'r') as f:
    content = f.read()

# Display it as a highlighted Bash block
display(Markdown(f"```bash\n{content}\n```"))

```bash
#!/bin/bash
## Written by Ye Kyaw Thu, Affiliated Professor, CADT, Cambodia
## for NMT Experiments between Burmese and Ethnic Languages
## used Marian NMT Framework for seq2seq training
## Last updated: 23 May 2022
## Reference: https://marian-nmt.github.io/examples/mtm2017/complex/
model_folder="model.seq2seq.phmy";
mkdir ${model_folder};
data_path="/home/phantom/Documents/assignment_7_testing/kaung-htet-htun_assignment-7/g2p-par";
src="ph"; tgt="my";
~/marian/build/marian \
--type s2s \
--train-sets ${data_path}/train.${src} ${data_path}/train.${tgt} \
--max-length 200 \
--valid-sets ${data_path}/dev.${src} ${data_path}/dev.${tgt} \
--vocabs ${data_path}/vocab/vocab.${src}.yml ${data_path}/vocab/vocab.${tgt}.yml \
--model ${model_folder}/model.npz \
--workspace 500 \
--enc-depth 2 --enc-type alternating --enc-cell lstm --enc-cell-depth 2 \
--dec-depth 2 --dec-cell lstm --dec-cell-base-depth 2 --dec-cell-high-depth 2 \
--tied-embeddings --layer-normalization --skip \
--mini-batch-fit \
--valid-mini-batch 32 \
--valid-metrics cross-entropy perplexity bleu \
--valid-freq 5000 --save-freq 5000 --disp-freq 500 \
--dropout-rnn 0.3 --dropout-src 0.3 --exponential-smoothing \
--early-stopping 10 \
--log ${model_folder}/train.log --valid-log ${model_folder}/valid.log \
--devices 0 --sync-sgd --seed 1111 \
--dump-config > ${model_folder}/config.yml
time ~/marian/build/marian -c ${model_folder}/config.yml
2>&1 | tee ${model_folder}/s2s.${src}-${tgt}.log
```

အရေးကြီးတဲ့ parameter တွေကိုပဲ ဆရာ သုံးပြထားပါတယ်။ ကျောင်းသားတွေက ကိုယ်စက်ထဲမှာ run မယ်ဆိုရင်တော့ အဓိက ပြင်ရမှာက path တွေပါပဲ။ 

```bash
model_folder="model.seq2seq.phmy";
mkdir ${model_folder};
data_path="/home/ye/exp/nmt/marian-demo/g2p-par";
src="ph"; tgt="my";
```

အာခီတက်ချာကိုတော့ type ဆိုတဲ့ option နဲ့ ပြောင်းလို့ ရပါတယ်။  

```bash
--type s2s 
```

ပြီးတော့ GPU က တစ်လုံးထက်ပိုရင် 0 1 2 3 ဆိုပြီး ဖြည့်တာမျိုး လုပ်ပေးရလိမ့်မယ်။ 0 to 3 ပေးထားရင် ကိုယ့်စက်ထဲမှာ ရှိတဲ့ GPU ကဒ်လေးကဒ်စလုံးကို အသုံးပြုမယ်လို့ ဆိုလိုတာပါ။  

အများသောအားဖြင့်က Neural network မော်ဒယ်တွေကို training လုပ်တဲ့အခါမှာ play ရတဲ့ batch size, memory ပိုင်း, hidden layer အရေအတွက်, epoch, dropout စတာတွေပဲ ဖြစ်ပါတယ်။ Marian framework မှာဆိုရင်တော့ အများသောအားဖြင့် error တက်တာက -workspace, --enc-depth, --decc-depth တွေကလည်း အရေးကြီးပါတယ်။ Learning လုပ်နေတဲ့အချိန်မှာ ဘယ်လို evaluation metric နဲ့ ထားမလဲ ဆိုတာကိုတော့ --valid-metrics ဆိုတဲ့ command line option နဲ့ ချိန်လို့ ရပါတယ်။ ဆရာကတော့ အထက်မှာ မြင်ရတဲ့အတိုင်း `--valid-metrics cross-entropy perplexity bleu` သုံးခု ထားပါတယ်။ ပြီးတော့ ကိုယ့် HDD ထဲမှာ မော်ဒယ်တွေ အများကြီးသိမ်းရင် hardisk space ယူတာမို့လို့ ဆရာကတော့ `-save-freq 5000` ဆိုတဲ့ setting ကို လုပ်ထားပါတယ်။ အခြေခံအားဖြင့် ဆရာပြင်ပေးထားတဲ့ setting ဖိုင်နဲ့ အဆင်ပြေပါလိမ့်မယ်။ သို့သော် ကိုယ့်စက်ရဲ့ GPU, memory ပြီးတော့ တခြား လိုချင်တဲ့ training/tuning parameter တွေကတော့ ကိုယ်တိုင် run ကြည့်လိုက် ရလဒ်ကို ကြည့်လိုက်လုပ်ပြီး ညှိယူရပါလိမ့်မယ်။ အထူးသဖြင့် ကိုယ့် ဒေတာပမာဏ၊ ဒိုမိန်းနဲ့ အကောင်းဆုံး NMT performance ကို ရဖို့အတွက်က တစ်ကြိမ်ထက်မက training လုပ်ကြရပါလိမ့်မယ်။    

Shell script မှာက နှစ်ပိုင်း ပါဝင်ပါတယ်။ တစ်ပိုင်း (သို့) ပထမ command က configuration ဖိုင် တည်ဆောက်တာပါ။ နောက်တစ်ပိုင်း က ဆောက်ထားတဲ့ configuration ဖိုင်ကို သုံးပြီးတော့ NMT လုပ်တဲ့ အပိုင်းပါ။  



## Seq2Seq Training for Phoneme to Grapheme


In [23]:
!./seq2seq.phmy.sh

[2026-07-02 21:26:49] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-07-02 21:26:49] [marian] Running on PS42 as process 1266247 with command line:
[2026-07-02 21:26:49] [marian] /home/phantom/marian/build/marian -c model.seq2seq.phmy/config.yml
[2026-07-02 21:26:49] [config] after: 0e
[2026-07-02 21:26:49] [config] after-batches: 0
[2026-07-02 21:26:49] [config] after-epochs: 0
[2026-07-02 21:26:49] [config] all-caps-every: 0
[2026-07-02 21:26:49] [config] allow-unk: false
[2026-07-02 21:26:49] [config] authors: false
[2026-07-02 21:26:49] [config] beam-size: 12
[2026-07-02 21:26:49] [config] bert-class-symbol: "[CLS]"
[2026-07-02 21:26:49] [config] bert-mask-symbol: "[MASK]"
[2026-07-02 21:26:49] [config] bert-masking-fraction: 0.15
[2026-07-02 21:26:49] [config] bert-sep-symbol: "[SEP]"
[2026-07-02 21:26:49] [config] bert-train-type-embeddings: true
[2026-07-02 21:26:49] [config] bert-type-vocab-size: 2
[2026-07-02 21:26:49] [config] build-info: ""
[2026-07-02 21:2

အထက်မှာ မြင်ရတဲ့အတိုင်း training လုပ်တာက ၂၃ မိနစ်အကြာမှာ ပြီးသွားပါတယ်။  
Epoch 164 မှာ early stop setting ကြောင့် ရပ်သွားပါတယ်။  
အကောင်းဆုံး BLEU score က 77.71 ရရှိပါတယ်။ 
Machine translation ရဲ့ performance အနေနဲ့ဆိုရင်တော့ ကောင်းပါတယ်။ ဒီရလဒ်က Validation set နဲ့ ရတဲ့ ရလဒ်ပါ။ တကယ်တမ်း ရလဒ်အနေနဲ့ ချပြလို့ ရတာကတော့ test data နဲ့ evaluation လုပ်ကြည့်ပြီး ရလာတဲ့ ရလဒ်ဖြစ်ပါလိမ့်မယ်။  

အရင်ဆုံး training လုပ်စဉ်မှာ သုံးသွားတဲ့ configuration yml ဖိုင်ကိုလည်း ဖွင့်ကြည့်ပြီး လေ့လာကြရအောင်။  

## Configuration File for Ph-My

In [24]:
!cat ./model.seq2seq.phmy/config.yml

# Marian configuration file generated at 2026-07-02 09:26:49 PM +0630 with version v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
# General options
authors: false
cite: false
build-info: ""
workspace: 500
log: model.seq2seq.phmy/train.log
log-level: info
log-time-zone: ""
quiet: false
quiet-translation: false
seed: 1111
check-nan: false
interpolate-env-vars: false
relative-paths: false
sigterm: save-and-exit
# Model options
model: model.seq2seq.phmy/model.npz
pretrained-model: ""
ignore-model-config: false
type: s2s
dim-vocabs:
  - 0
  - 0
dim-emb: 512
factors-dim-emb: 0
factors-combine: sum
lemma-dependency: ""
lemma-dim-emb: 0
dim-rnn: 1024
enc-type: alternating
enc-cell: lstm
enc-cell-depth: 2
enc-depth: 2
dec-cell: lstm
dec-cell-base-depth: 2
dec-cell-high-depth: 2
dec-depth: 2
skip: true
layer-normalization: true
right-left: false
input-types:
  []
best-deep: false
tied-embeddings: true
tied-embeddings-src: false
tied-embeddings-all: false
output-omit-bias: false
transformer-heads: 8


## Checking Output Models

In [25]:
!ls ./model.seq2seq.phmy/ --color=auto

config.yml          model.npz.decoder.yml    model.npz.yml  valid.log
model.iter5000.npz  model.npz.optimizer.npz  s2s.ph-my.log
model.npz           model.npz.progress.yml   train.log


အထက်မှာ မြင်ရတဲ့ npz ဖိုင်တွေက မော်ဒယ် ဖိုင်တွေပါပဲ။ ဆရာက iteration 5000 စီမှာ တစ်ခေါက်စီ သိမ်းထားတဲ့ setting နဲ့ run ခဲ့တာပါ။ အဲဒီအထဲကမှ training process တစ်ခုလုံးအတွက် အကောင်းဆုံး မော်ဒယ်ဖိုင်က model.npz ဖိုင်ပါပဲ။  

training log ဖိုင်တွေ validation log ဖိုင်တွေကိုလည်း လေ့လာကြည့်ပါ။  

## Testing with Seq2Seq Model for Phoneme to Grapheme Translation

အရင်ဆုံး GPU က အားမအားလည်း စစ်ကြည့်ပါ။  

In [26]:
!nvidia-smi

Thu Jul  2 22:14:54 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.309.01             Driver Version: 535.309.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce GTX 1050 ...    Off | 00000000:03:00.0 Off |                  N/A |
| N/A   56C    P8              N/A / ERR! |      4MiB /  4096MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [4]:
!time ~/marian/build/marian-decoder \
  -m ./model.seq2seq.phmy/model.npz \
  -v ../g2p-par/vocab/vocab.ph.yml ../g2p-par/vocab/vocab.my.yml \
 --devices 0 \
  < ../g2p-par/test.ph \
  > ../seq2seq.phmy.hyp.txt

[2026-07-03 20:46:47] Error: Model file does not exist: ./model.seq2seq.phmy/model.npz
[2026-07-03 20:46:47] Error: Aborted from void marian::ConfigValidator::validateOptionsTranslation() const in /home/phantom/marian/src/common/config_validator.cpp:64

[CALL STACK]
[0x5594c0f0b799]    marian::ConfigValidator::  validateOptionsTranslation  () const + 0x909
[0x5594c0f150c6]    marian::ConfigValidator::  validateOptions  (marian::cli::mode) const + 0x46
[0x5594c0ed0f45]    marian::ConfigParser::  parseOptions  (int,  char**,  bool) + 0x305
[0x5594c0eb4ad5]    marian::  parseOptions  (int,  char**,  marian::cli::mode,  bool) + 0x55
[0x5594c0ddd84d]    main                                               + 0x3d
[0x7a231ea29d90]                                                       + 0x29d90
[0x7a231ea29e40]    __libc_start_main                                  + 0x80
[0x5594c0e1c675]    _start                                             + 0x25

/bin/bash: line 1: 36045 Aborted               

## Checking Hypothesis File  

Test data နဲ့ translation လုပ်ပြီး ရလာတဲ့ hypothesis ဖိုင်ကို လေ့လာကြည့်ကြရအောင်။  

In [1]:
!head -n 30  /home/phantom/Documents/assignment_7_testing/kaung-htet-htun_assignment-7/seq2seq.phmy.hyp.txt

တက် တက် ပြောင်
ကပ် ပိ
ရှုံ့ မဲ့
ညှဉ်း ပန်း
မွန်း မံ
ငယ် မည်
လျှာ ရှာ ဖွေ ဗွေ
ဘ ပြင်း
ဥ မ ကွဲ သိုက် မ ပျက်
အ စစ်
ရေ ကျ
မ ဆုတ် မ ဆိုင်း
ဟ ခွက်
မိန်း မူး
ကယ် မ
နင်း နယ်
က ကြိုး တံ စာ
ဆွမ်း ကြီး လောင်း
ဝါ ကျင့် ကျင့်
ကုပ် ဟဲ နာ
နီ ကြင် ကြင်
အ လို တူ
ကိန်း ရင်း
မီး တိုင်
ဝေ့ လည် ကြောင် ဖတ်
စ ကား ပြော ကြေး နန်း
အ ချစ် ဦး
အ သည်း ကောင်း
ည ကြီး
ချို ချို မြ


**Test input ဖိုင်နဲ့ တွဲကြည့်ကြည့်ရအောင်။**  

In [9]:
!paste ../g2p-par/test.ph ../seq2seq.phmy.hyp.txt | head -n 30 

te' te' pjaun	တက် တက် ပြောင်
ka' pi.	ကပ် ပိ
shoun. me.	ရှုံ့ မဲ့
njhin: ban:	ညှဉ်း ပန်း
mun: man	မွန်း မံ
nge mji	ငယ် မည်
sha sha hpwei bwei	လျှာ ရှာ ဖွေ ဗွေ
ba- bjin:	ဘ ပြင်း
u. ma- kwe: thai' ma- pje'	ဥ မ ကွဲ သိုက် မ ပျက်
a- si'	အ စစ်
jei kya.	ရေ ကျ
ma- hsou' ma- hsain:	မ ဆုတ် မ ဆိုင်း
ha. gwe'	ဟ ခွက်
mein: mu:	မိန်း မူး
ke ma.	ကယ် မ
nin: ne	နင်း နယ်
ka. gyou: da- za	က ကြိုး တံ စာ
hsun: gyi: laun:	ဆွမ်း ကြီး လောင်း
wa kyin. gyin.	ဝါ ကျင့် ကျင့်
kou' he: na	ကုပ် ဟဲ နာ
ni kyin gyin	နီ ကြင် ကြင်
a- lou tu	အ လို တူ
kein: jin:	ကိန်း ရင်း
mi: dain	မီး တိုင်
wei. le gyaun ba'	ဝေ့ လည် ကြောင် ဖတ်
za- ga: bjo: kyei: nan:	စ ကား ပြော ကြေး နန်း
a- chi' u:	အ ချစ် ဦး
a- the: kaun:	အ သည်း ကောင်း
nja. gyi:	ည ကြီး
chou mja mja.	ချို ချို မြ
paste: write error: Broken pipe
paste: write error


## Evaluation on Seq2Seq Model 

SMT demo တုန်းက သုံးခဲ့တဲ့ BLEU socre တွက်တဲ့ perl script ကိုပဲ သုံးပြီး evaluation လုပ်ကြည့်ပါမယ်။  

In [10]:
!perl /home/phantom/mosesdecoder/scripts/generic/multi-bleu.perl ../g2p-par/test.my < ../seq2seq.phmy.hyp.txt

BLEU = 74.20, 86.3/76.4/70.4/65.6 (BP=0.999, ratio=0.999, hyp_len=8041, ref_len=8047)
It is not advisable to publish scores from multi-bleu.perl.  The scores depend on your tokenizer, which is unlikely to be reproducible from your paper or consistent across research groups.  Instead you should detokenize then use mteval-v14.pl, which has a standard tokenization.  Scores from multi-bleu.perl can still be used for internal purposes when you have a consistent tokenizer.
